# 03 — Backtest & Regression: Forward Returns by Quartile

**Concentric Value Model · Empirical Validation · DBA Research**

**Research question Q-A:** *Do CVM top-quartile firms deliver higher risk-adjusted forward returns than bottom-quartile firms over 2016–2025?*

**Specification:** Forward 1-year total return regressed on CVM composite (or quartile dummies), with **all five controls**:
- Sector fixed effects
- Country fixed effects
- Log market cap (size factor)
- Market beta (CAPM-style)
- Regime dummies: COVID-2020 and 2022-2023 inflation/hiking

**Standard errors:** clustered by `as_of` quarter (accounts for cross-sectional return correlation within quarters).

**Outputs to Google Drive:**
- `quartile_portfolio_returns.csv` — Q1/Q2/Q3/Q4 mean returns per quarter
- `quartile_spread_series.csv` — Q1 minus Q4 time series
- `regression_continuous.csv` — full continuous-composite regression
- `regression_quartile_dummies.csv` — quartile-dummy regression (headline)
- `framework_alpha_summary.csv` — single-row result for thesis abstract
- `framework_alpha_by_period.csv` — sub-period robustness
- 2 figures (quartile growth paths, sub-period alpha)

## Step 1 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = '/content/drive/MyDrive/cvm-research'
# Resolve the study window from the metadata nb01 wrote — never hardcode the
# filename, so changing START_YEAR/END_YEAR in nb01 can't silently desync.
import json as _json
with open(f'{WORK_DIR}/output/panel_metadata.json') as _f:
    _meta = _json.load(_f)
_window = _meta['window']  # e.g. '2016-2025'
_ys, _ye = _window.split('-')
PANEL_FILE = f"{WORK_DIR}/output/panel_pit_{_ys}_{_ye}.parquet"
SCORED_FILE = f"{WORK_DIR}/output/scored_panel_{_ys}_{_ye}.parquet"
print(f'Study window from metadata: {_window}')
scored_path = SCORED_FILE
assert os.path.exists(scored_path), 'Run 02_pca_analysis.ipynb first (it saves the scored panel).'

REPO_PATH = 'FelixDLanger/cvm-research'
import sys
!pip install statsmodels --quiet
!git clone https://github.com/{REPO_PATH}.git /content/cvm-research 2>/dev/null || (cd /content/cvm-research && git pull)
sys.path.insert(0, '/content/cvm-research')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from cvm_research.scoring import DIMENSIONS, LAYERS
from cvm_research.stats import (
    build_quartile_portfolios, quartile_spread_series, add_regime_dummies,
    run_panel_regression, run_quartile_dummy_regression,
    annualized_sharpe, regression_table_summary, sig_stars,
)
import cvm_research
print(f'cvm_research v{cvm_research.__version__}')

## Step 2 — Load scored panel and prepare for regression

In [ ]:
scored = pd.read_parquet(scored_path)
print(f'Loaded {len(scored):,} firm-quarter observations.')

# Add regime dummies
scored = add_regime_dummies(scored, date_col='as_of')
print()
print('Regime composition:')
print(f'  COVID regime (2020):       {scored["covid_regime"].sum():,} obs ({100*scored["covid_regime"].mean():.1f}%)')
print(f'  Inflation regime (2022-23): {scored["inflation_regime"].sum():,} obs ({100*scored["inflation_regime"].mean():.1f}%)')
print()
print('Coverage of regression inputs:')
for col in ['composite', 'forward_return_pct', 'market_cap', 'beta', 'sector', 'country']:
    n_present = scored[col].notna().sum() if col in scored.columns else 0
    print(f'  {col:22s}: {n_present:5d} / {len(scored):5d} ({100*n_present/len(scored):4.1f}%)')

## Step 3 — Quartile portfolio descriptives

In [ ]:
qport = build_quartile_portfolios(scored, return_col='forward_return_pct')
qport_pivot = qport.pivot(index='as_of', columns='quartile', values='mean_return')

# Full-sample summary
print('Mean forward 1-yr return by quartile (full sample):')
overall = qport.groupby('quartile')['mean_return'].agg(['mean','std','count']).round(2)
overall['sharpe_annualised'] = overall.apply(
    lambda r: round(r['mean']/r['std']*np.sqrt(4), 3) if r['std'] > 0 else np.nan, axis=1)
print(overall)
print()

spread = quartile_spread_series(qport)
ann_mean = spread['Q1_minus_Q4'].mean() * 4
ann_vol = spread['Q1_minus_Q4'].std() * np.sqrt(4)
print(f'Annualised Q1-minus-Q4 spread:')
print(f'  Mean:       {ann_mean:+.2f}%')
print(f'  Volatility: {ann_vol:.2f}%')
if ann_vol > 0:
    print(f'  Sharpe:     {ann_mean/ann_vol:+.3f}')

## Step 4 — Visualise quartile portfolio growth paths

In [ ]:
# Cumulative growth: convert quarterly % returns to growth factors, then cumprod
quartile_paths = qport_pivot.copy().sort_index()
for q in ['Q1','Q2','Q3','Q4']:
    if q in quartile_paths.columns:
        quartile_paths[f'{q}_growth'] = (1 + quartile_paths[q]/100).cumprod()

fig, ax = plt.subplots(figsize=(12, 6))
colors = {'Q1':'#4a7c3a', 'Q2':'#a8814a', 'Q3':'#888777', 'Q4':'#a14b3a'}
for q in ['Q1','Q2','Q3','Q4']:
    col = f'{q}_growth'
    if col in quartile_paths.columns:
        ax.plot(quartile_paths.index, quartile_paths[col], label=q,
                color=colors[q], linewidth=2)
ax.axhline(1.0, color='gray', linewidth=0.5, linestyle='--')
ax.set_xlabel('Date')
ax.set_ylabel('Cumulative Growth (1.0 = starting value)')
ax.set_title('Quartile Portfolio Growth Paths\n1-Year Forward Total Return, Compounded Quarterly')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{WORK_DIR}/output/quartile_growth_paths.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 5 — Headline regression (continuous composite + all 5 controls)

In [ ]:
# Specification: forward_return ~ composite + C(sector) + C(country)
#                                  + log_mcap + beta + covid_regime + inflation_regime
# Cluster-robust SE by as_of quarter
result_continuous = run_panel_regression(
    scored,
    return_col='forward_return_pct',
    composite_col='composite',
    cluster_by='as_of',
    verbose=True,
)
print(result_continuous.summary())

**Interpretation:**

- The `composite` coefficient is the marginal effect of a 1-point increase in CVM composite on forward 1-yr return (%), holding sector, country, size, beta, and regime constant.
- E.g. coef = 0.15 means: a 10-point higher CVM composite → +1.5% higher forward return, controls held fixed.
- p-value < 0.05 → statistically distinguishable from zero.
- Standard errors clustered by `as_of` quarter, accounting for cross-sectional correlation in forward returns within each quarter.

## Step 6 — Quartile dummy regression (Q1 vs Q4 — headline result)

In [ ]:
result_quartile = run_quartile_dummy_regression(
    scored,
    return_col='forward_return_pct',
    reference_quartile='auto',
    cluster_by='as_of',
    verbose=True,
)
print(result_quartile.summary())

In [ ]:
# Extract & interpret quartile coefficients (reference-aware)
ref_q = getattr(result_quartile, '_cvm_reference_quartile', 'Q4')
summary_df = regression_table_summary(result_quartile)

# All quartile dummy rows
qrows = summary_df[summary_df['variable'].str.contains('quartile', na=False)].copy()

print('='*64)
print(f'QUARTILE COEFFICIENTS  (reference group = {ref_q})')
print('='*64)
print(f'Each coefficient = mean forward-return difference vs {ref_q}, controls held fixed.\n')

import re
for _, row in qrows.iterrows():
    m = re.search(r'T\.(Q\d)', row['variable'])
    label = m.group(1) if m else row['variable']
    print(f'  {label} vs {ref_q}:  {row["coefficient"]:+7.3f}%   '
          f'(SE {row["std_error"]:.2f}, p={row["p_value"]:.4f} {sig_stars(row["p_value"])}, '
          f'95% CI [{row["ci_lower"]:+.2f}, {row["ci_upper"]:+.2f}])')

print('='*64)
print('INTERPRETATION GUIDE')
print('='*64)
print(f"""
The framework's directional claim is: HIGHER quartile (Q1 = High Conviction)
should outperform LOWER quartile (Q4 = Structural Concerns).

  • If Q1's coefficient is the LARGEST positive vs {ref_q}: framework ranks correctly.
  • If a LOWER quartile (Q2/Q3) beats Q1: the ranking is INVERTED in this sample
    — investigate before interpreting. Common causes: thin/regime-restricted
    sample, baseline-dominated scoring (check nb02 coverage), or genuine
    contrarian effect (rare; needs robustness checks across sub-periods).

Quote in the thesis with full uncertainty:
  "Relative to {ref_q} firms, Q1 firms showed X.XX% (95% CI [a,b], p=...)
   higher 1-year forward returns, controlling for sector, country, log market
   cap, beta, and regime."
""")


**Quoting this in your thesis abstract:**

> *"After controlling for sector, country, log market capitalisation, market beta, and regime dummies for COVID-19 (2020) and the 2022–2023 inflation/hiking cycle, top-quartile firms identified by the Concentric Value Model delivered X.XX% (95% CI: [a, b], p<X.XX) higher 1-year forward returns than bottom-quartile firms across the 2016–2025 sample period (N=X firm-quarter observations, standard errors clustered by quarter-end date)."*

Always quote the **point estimate with its 95% CI and p-value**, not the point estimate alone. This is the rigor a DBA examiner expects.

## Step 7 — Sub-period robustness

In [ ]:
periods = {
    'Pre-COVID (2016-2019)':     scored['as_of'].between('2016-01-01','2019-12-31'),
    'COVID era (2020-2021)':     scored['as_of'].between('2020-01-01','2021-12-31'),
    'Inflation+ (2022-2025)':    scored['as_of'].between('2022-01-01','2025-12-31'),
}

period_results = []
for name, mask in periods.items():
    sub = scored[mask].copy()
    n_obs = sub['forward_return_pct'].notna().sum()
    if n_obs < 50:
        print(f'  Skipping {name}: only {n_obs} observations')
        continue
    try:
        r = run_quartile_dummy_regression(sub, cluster_by='as_of', verbose=False)
        sd = regression_table_summary(r)
        q1 = sd[sd['variable'].str.contains('Q1', na=False) &
                sd['variable'].str.contains('quartile', na=False)]
        if len(q1) > 0:
            row = q1.iloc[0]
            period_results.append({
                'period': name,
                'n_obs': n_obs,
                'q1_alpha': round(row['coefficient'], 3),
                'std_err':  round(row['std_error'], 3),
                'p_value':  round(row['p_value'], 4),
                'sig':      sig_stars(row['p_value']),
                'ci_lower': round(row['ci_lower'], 3),
                'ci_upper': round(row['ci_upper'], 3),
            })
    except Exception as e:
        print(f'  {name} failed: {e}')

period_df = pd.DataFrame(period_results)
print()
print('Framework alpha across sub-periods:')
print(period_df.to_string(index=False))

In [ ]:
# Visualise sub-period alphas with error bars
if len(period_results) > 0:
    fig, ax = plt.subplots(figsize=(9, 5))
    x = range(len(period_df))
    ax.bar(x, period_df['q1_alpha'], color='#a8814a', alpha=0.85)
    ax.errorbar(x, period_df['q1_alpha'],
                yerr=[period_df['q1_alpha'] - period_df['ci_lower'],
                      period_df['ci_upper'] - period_df['q1_alpha']],
                fmt='none', color='black', capsize=5)
    ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
    ax.set_xticks(x)
    ax.set_xticklabels(period_df['period'], rotation=0, fontsize=9)
    ax.set_ylabel('Framework Alpha (% per 1-yr forward)')
    ax.set_title('Q1-minus-Q4 Framework Alpha by Sub-Period\n(95% CI shown; controls: sector, country, log mcap, beta, regime)')
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(f'{WORK_DIR}/output/framework_alpha_by_period.png', dpi=150, bbox_inches='tight')
    plt.show()

## Step 8 — Save all outputs for thesis appendix

In [ ]:
# Quartile portfolio data
qport.to_csv(f'{WORK_DIR}/output/quartile_portfolio_returns.csv', index=False)
spread.to_csv(f'{WORK_DIR}/output/quartile_spread_series.csv', index=False)

# Regression tables
regression_table_summary(result_continuous).to_csv(
    f'{WORK_DIR}/output/regression_continuous.csv', index=False)
regression_table_summary(result_quartile).to_csv(
    f'{WORK_DIR}/output/regression_quartile_dummies.csv', index=False)

# Framework alpha summary (reference-aware: derives reference from result_quartile)
ref_q = getattr(result_quartile, '_cvm_reference_quartile', 'Q4')
qsum = regression_table_summary(result_quartile)
q1_rows = qsum[qsum['variable'].str.contains('Q1', na=False) &
               qsum['variable'].str.contains('quartile', na=False)]

if len(q1_rows) > 0:
    row = q1_rows.iloc[0]
    pd.DataFrame({
        'metric': [f'Framework Alpha (Q1 vs {ref_q})'],
        'coefficient_pct': [row['coefficient']],
        'std_error': [row['std_error']],
        't_stat': [row['t_stat']],
        'p_value': [row['p_value']],
        'significance': [sig_stars(row['p_value'])],
        'ci_lower': [row['ci_lower']],
        'ci_upper': [row['ci_upper']],
        'n_obs': [int(result_quartile.nobs)],
        'reference_quartile': [ref_q],
        'sample_period': ['2016-Q1 to 2025-Q4'],
        'standard_errors': ['cluster-robust by as_of quarter'],
        'controls': ['sector FE, country FE, log market_cap, beta, covid_regime, inflation_regime'],
    }).to_csv(f'{WORK_DIR}/output/framework_alpha_summary.csv', index=False)

# Sub-period table
if len(period_results) > 0:
    period_df.to_csv(f'{WORK_DIR}/output/framework_alpha_by_period.csv', index=False)

print('All regression outputs saved to:', f'{WORK_DIR}/output/')


## Summary — Q-A Answer

You now have the empirical answer to the headline DBA research question. The framework alpha is in `framework_alpha_summary.csv`.

**Three possible outcomes and how to frame each in the thesis:**

1. **Significant positive alpha (p<0.05):** The framework identifies firms that systematically outperform after rich controls. The thesis claims empirical validity for the methodology.

2. **Significant negative alpha:** The framework's top-quartile firms underperform. Itself an important finding — the framework as published would be a contrarian indicator.

3. **Not significant (p>0.10):** Most likely outcome at this sample size and rigor. The thesis argues the framework's contribution is as a **thinking tool** that organizes and structures the analysis of company quality across dimensions, independent of return-prediction claims. The PCA validation (notebook 02) speaks to the structural validity of the framework's dimensions; the regression speaks to its return-prediction power, and these can be reported separately.

**For viva defense**, the sub-period table shows robustness — examiners will ask "does this hold in normal markets, not just COVID?" The breakdown answers that directly.

**Combined with Q-B from notebook 02**, you have a complete empirical chapter: PCA confirms (or refines) the layer weights, AND quartile membership correlates (or doesn't) with forward returns after rigorous controls. That's a defensible chapter.

---

# US-Only Analysis (Primary Empirical Test)

**Rationale.** The global panel includes ~100 US tickers (full ownership data via yfinance) and ~100 non-US tickers (D6/D7/D9 on sector baseline). Non-US firms therefore have noisier quartile assignments. Restricting to US firms isolates the framework from data-coverage heterogeneity and provides the cleanest empirical test.

**Thesis framing.** This US-only analysis serves as the PRIMARY empirical test in the chapter. The global pooled result above serves as a ROBUSTNESS CHECK showing how findings change when expanded to a heterogeneously-covered universe.

All controls, regime dummies, and clustering are identical to the global analysis — only the sample is restricted.

In [ ]:
# Restrict to US firms only
scored_us = scored[scored['country'] == 'US'].copy()
print(f'US-only sample: {len(scored_us):,} firm-quarters '
      f'({scored_us["ticker"].nunique()} unique tickers)')
print(f'Date range: {scored_us["as_of"].min()} to {scored_us["as_of"].max()}')
print()
print('Quartile distribution (US-only):')
print(scored_us['quartile'].value_counts().sort_index())
print()
print('Forward-return coverage:')
fr_us = scored_us['forward_return_pct'].notna()
print(f'  non-null: {fr_us.sum():,} / {len(scored_us):,} ({100*fr_us.mean():.1f}%)')


In [ ]:
# US-only continuous regression — full controls (country FE dropped since all US)
print('US-ONLY CONTINUOUS REGRESSION')
print('='*64)
result_continuous_us = run_panel_regression(
    scored_us,
    return_col='forward_return_pct',
    composite_col='composite',
    cluster_by='as_of',
)
print(result_continuous_us.summary())


In [ ]:
# US-only quartile-dummy regression
print('US-ONLY QUARTILE-DUMMY REGRESSION')
print('='*64)
result_quartile_us = run_quartile_dummy_regression(
    scored_us,
    return_col='forward_return_pct',
    reference_quartile='auto',
    cluster_by='as_of',
)
print(result_quartile_us.summary())


In [ ]:
# US-only reference-aware quartile coefficient summary
ref_q_us = getattr(result_quartile_us, '_cvm_reference_quartile', 'Q4')
qsum_us = regression_table_summary(result_quartile_us)
qrows_us = qsum_us[qsum_us['variable'].str.contains('quartile', na=False)].copy()

print('='*64)
print(f'US-ONLY QUARTILE COEFFICIENTS  (reference = {ref_q_us})')
print('='*64)
import re
for _, row in qrows_us.iterrows():
    m = re.search(r'T\\.(Q\\d)', row['variable'])
    label = m.group(1) if m else row['variable']
    print(f'  {label} vs {ref_q_us}:  {row["coefficient"]:+7.3f}%   '
          f'(SE {row["std_error"]:.2f}, p={row["p_value"]:.4f} {sig_stars(row["p_value"])}, '
          f'95% CI [{row["ci_lower"]:+.2f}, {row["ci_upper"]:+.2f}])')
print('='*64)
print(f'N observations: {int(result_quartile_us.nobs)}')
print(f'R-squared: {result_quartile_us.rsquared:.3f}')


In [ ]:
# US-only sub-period analysis (same windows as global)
import matplotlib.pyplot as plt

scored_us['as_of'] = pd.to_datetime(scored_us['as_of'])
periods_us = {
    'Pre-COVID (2016-2019)':     scored_us['as_of'].between('2016-01-01','2019-12-31'),
    'COVID era (2020-2021)':     scored_us['as_of'].between('2020-01-01','2021-12-31'),
    'Inflation+ (2022-2025)':    scored_us['as_of'].between('2022-01-01','2025-12-31'),
}

period_results_us = []
for label, mask in periods_us.items():
    sub = scored_us[mask].copy()
    if len(sub) < 100 or sub['quartile'].nunique() < 2:
        print(f'{label}: insufficient data (n={len(sub)}, quartiles={sub["quartile"].nunique()})')
        continue
    try:
        r = run_quartile_dummy_regression(sub, cluster_by='as_of', verbose=False)
        ref = getattr(r, '_cvm_reference_quartile', 'Q4')
        qsum = regression_table_summary(r)
        q1 = qsum[qsum['variable'].str.contains('Q1', na=False) &
                  qsum['variable'].str.contains('quartile', na=False)]
        if len(q1) > 0:
            row = q1.iloc[0]
            period_results_us.append({
                'period': label, 'reference': ref, 'n_obs': int(r.nobs),
                'q1_vs_ref_coef': row['coefficient'],
                'std_error': row['std_error'],
                'p_value': row['p_value'],
                'ci_lower': row['ci_lower'], 'ci_upper': row['ci_upper'],
            })
            print(f'{label}: Q1 vs {ref} = {row["coefficient"]:+.2f}% '
                  f'(SE {row["std_error"]:.2f}, p={row["p_value"]:.4f}, n={int(r.nobs)})')
    except Exception as e:
        print(f'{label}: regression failed — {e}')

if period_results_us:
    period_df_us = pd.DataFrame(period_results_us)
    print()
    print(period_df_us.to_string(index=False))


In [ ]:
# Side-by-side comparison: global vs US-only headline results
ref_global = getattr(result_quartile, '_cvm_reference_quartile', 'Q4')
qsum_global = regression_table_summary(result_quartile)
q1_global = qsum_global[qsum_global['variable'].str.contains('Q1', na=False) &
                        qsum_global['variable'].str.contains('quartile', na=False)]
q1_us = qsum_us[qsum_us['variable'].str.contains('Q1', na=False) &
                qsum_us['variable'].str.contains('quartile', na=False)]

comparison_rows = []
if len(q1_global) > 0:
    rg = q1_global.iloc[0]
    comparison_rows.append({
        'sample': 'Global (200 firms)', 'n_obs': int(result_quartile.nobs),
        'reference': ref_global, 'q1_alpha_pct': rg['coefficient'],
        'std_error': rg['std_error'], 'p_value': rg['p_value'],
        'sig': sig_stars(rg['p_value']),
    })
if len(q1_us) > 0:
    ru = q1_us.iloc[0]
    comparison_rows.append({
        'sample': 'US only (~100 firms)', 'n_obs': int(result_quartile_us.nobs),
        'reference': ref_q_us, 'q1_alpha_pct': ru['coefficient'],
        'std_error': ru['std_error'], 'p_value': ru['p_value'],
        'sig': sig_stars(ru['p_value']),
    })

comparison_df = pd.DataFrame(comparison_rows)
print('='*64)
print('PRIMARY (US-only) vs ROBUSTNESS (Global) — Framework Alpha')
print('='*64)
print(comparison_df.to_string(index=False))
comparison_df.to_csv(f'{WORK_DIR}/output/us_vs_global_comparison.csv', index=False)

# Save US-only outputs
regression_table_summary(result_continuous_us).to_csv(
    f'{WORK_DIR}/output/regression_continuous_us.csv', index=False)
regression_table_summary(result_quartile_us).to_csv(
    f'{WORK_DIR}/output/regression_quartile_dummies_us.csv', index=False)
if period_results_us:
    pd.DataFrame(period_results_us).to_csv(
        f'{WORK_DIR}/output/framework_alpha_by_period_us.csv', index=False)

print()
print('US-only outputs saved.')


---

# Methodology Audit (Robustness Checks for Thesis Defensibility)

Six diagnostic checks that probe whether the empirical results above are robust or driven by confounds. Each check produces output that should be reported in the thesis methodology section and discussed if the result is sensitive.

1. **Top-20 by composite** — is the framework actually picking 'quality' or just mega-caps?
2. **Correlation matrix** — does composite collinearly track size/beta (signal absorption risk)?
3. **Beta-control robustness** — does the composite signal change when beta is removed?
4. **Size-control robustness** — same question for log market cap.
5. **Sector-fixed-effect robustness** — does the result depend on the sector controls?
6. **Equal-weighted vs cap-weighted quartile returns** — does the headline quartile result hold when each firm contributes equally vs proportional to its size?


In [ ]:
# AUDIT 1 — Top 20 firms by composite score (US-only)
# If these are all mega-cap tech/AI names, the framework is partly measuring 'is this a mega-cap'
# rather than 'is this genuinely well-managed'. Cap-weighted index effects then explain Q1 returns.

print('='*70)
print('AUDIT 1 — Top 20 firms by mean composite (US-only sample)')
print('='*70)

top20 = (scored_us.groupby('ticker')
         .agg(mean_composite=('composite','mean'),
              mean_mcap_bn=('market_cap', lambda s: s.mean()/1e9),
              n_obs=('composite','count'),
              sector=('sector','first'),
              quartile_mode=('quartile', lambda s: s.mode().iloc[0] if len(s.mode()) else None))
         .sort_values('mean_composite', ascending=False)
         .head(20)
         .round(1))
print(top20.to_string())

# Flag: how many of the top 20 are in the 'AI mega-cap' bucket?
ai_megacaps = {'NVDA','MSFT','AAPL','GOOGL','GOOG','AMZN','META','TSLA','AVGO','ORCL','LLY','BRK-B'}
ai_count = sum(1 for t in top20.index if t in ai_megacaps)
print(f'\nOf the top 20 by composite, {ai_count} are well-known AI/mega-cap names.')
if ai_count >= 10:
    print('FLAG: composite is heavily concentrated in mega-cap winners — interpret Q1 returns with caution.')
else:
    print('OK: composite is reasonably diversified across firm types.')


In [ ]:
# AUDIT 2 — Correlation matrix of composite vs size, beta, and forward return
# High correlations (>0.4) mean the composite carries the same signal as a standard control,
# so its 'unique' contribution to forward-return prediction is diluted by collinearity.

import numpy as np
print('='*70)
print('AUDIT 2 — Correlation matrix (US-only)')
print('='*70)

df_corr = scored_us[['composite','market_cap','beta','forward_return_pct']].copy()
df_corr['log_mcap'] = np.log(df_corr['market_cap'].replace(0, np.nan))
corr = df_corr[['composite','log_mcap','beta','forward_return_pct']].corr().round(3)
print(corr.to_string())

comp_mcap = corr.loc['composite','log_mcap']
comp_beta = corr.loc['composite','beta']
comp_fwd = corr.loc['composite','forward_return_pct']
print(f'\nKey correlations:')
print(f'  composite × log_mcap: {comp_mcap:+.3f}   {"FLAG: high collinearity with size" if abs(comp_mcap)>0.4 else "OK"}')
print(f'  composite × beta:     {comp_beta:+.3f}   {"FLAG: high collinearity with beta" if abs(comp_beta)>0.4 else "OK"}')
print(f'  composite × fwd_ret:  {comp_fwd:+.3f}   (raw — uncontrolled correlation with the dependent var)')


In [ ]:
# AUDIT 3 — Robustness: does composite signal change when BETA is dropped?
# If beta is absorbing the framework's signal (because high-quality firms tend to be high-beta
# in this AI-rally sample), removing beta should make composite significant.

import numpy as np
import statsmodels.formula.api as smf

print('='*70)
print('AUDIT 3 — Continuous regression WITHOUT beta control (US-only)')
print('='*70)

df = scored_us.dropna(subset=['forward_return_pct','composite','market_cap']).copy()
df['log_mcap'] = np.log(df['market_cap'].replace(0, np.nan))
df = df.dropna(subset=['log_mcap'])

# Without beta
formula_no_beta = 'forward_return_pct ~ composite + C(sector) + log_mcap + covid_regime + inflation_regime'
res_no_beta = smf.ols(formula_no_beta, data=df).fit(
    cov_type='cluster', cov_kwds={'groups': df['as_of']})
c1 = res_no_beta.params.get('composite')
p1 = res_no_beta.pvalues.get('composite')
se1 = res_no_beta.bse.get('composite')
print(f'Composite (no beta):        coef = {c1:+.4f}   SE = {se1:.4f}   p = {p1:.4f}')

# Reference: with beta (the original model)
formula_with_beta = 'forward_return_pct ~ composite + C(sector) + log_mcap + beta + covid_regime + inflation_regime'
res_with_beta = smf.ols(formula_with_beta, data=df.dropna(subset=['beta'])).fit(
    cov_type='cluster', cov_kwds={'groups': df.dropna(subset=['beta'])['as_of']})
c2 = res_with_beta.params.get('composite')
p2 = res_with_beta.pvalues.get('composite')
se2 = res_with_beta.bse.get('composite')
print(f'Composite (with beta, ref): coef = {c2:+.4f}   SE = {se2:.4f}   p = {p2:.4f}')

delta = c1 - c2
print(f'\nDelta when beta removed: {delta:+.4f}')
if abs(delta) > 0.2 or (p1 < 0.05) != (p2 < 0.05):
    print('FLAG: composite signal is sensitive to beta control. Beta is absorbing/displacing CVM signal.')
else:
    print('OK: composite signal is robust to beta control.')


In [ ]:
# AUDIT 4 — Robustness: does composite signal change when LOG_MCAP is dropped?
# If size is absorbing the framework's signal (because high-CVM firms are mostly mega-caps),
# removing log_mcap should make composite significant.

import statsmodels.formula.api as smf

print('='*70)
print('AUDIT 4 — Continuous regression WITHOUT log_mcap control (US-only)')
print('='*70)

formula_no_mcap = 'forward_return_pct ~ composite + C(sector) + beta + covid_regime + inflation_regime'
df_a4 = scored_us.dropna(subset=['forward_return_pct','composite','beta']).copy()
res_no_mcap = smf.ols(formula_no_mcap, data=df_a4).fit(
    cov_type='cluster', cov_kwds={'groups': df_a4['as_of']})
c1 = res_no_mcap.params.get('composite')
p1 = res_no_mcap.pvalues.get('composite')
se1 = res_no_mcap.bse.get('composite')
print(f'Composite (no log_mcap):       coef = {c1:+.4f}   SE = {se1:.4f}   p = {p1:.4f}')
print(f'Composite (with log_mcap, ref): coef = {c2:+.4f}   SE = {se2:.4f}   p = {p2:.4f}')
delta = c1 - c2
print(f'\nDelta when log_mcap removed: {delta:+.4f}')
if abs(delta) > 0.2 or (p1 < 0.05) != (p2 < 0.05):
    print('FLAG: composite signal is sensitive to size control. Size is absorbing/displacing CVM signal.')
else:
    print('OK: composite signal is robust to size control.')


In [ ]:
# AUDIT 5 — Robustness: does composite signal change when SECTOR FE is dropped?
# If the framework is partly measuring sector identity (because sector-tier dimensions exist
# for D8/D9/non-US D6/D7), removing sector FE could amplify the apparent signal — or remove it.

import statsmodels.formula.api as smf

print('='*70)
print('AUDIT 5 — Continuous regression WITHOUT sector FE (US-only)')
print('='*70)

formula_no_sector = 'forward_return_pct ~ composite + log_mcap + beta + covid_regime + inflation_regime'
df_a5 = scored_us.dropna(subset=['forward_return_pct','composite','beta','market_cap']).copy()
import numpy as np
df_a5['log_mcap'] = np.log(df_a5['market_cap'].replace(0, np.nan))
df_a5 = df_a5.dropna(subset=['log_mcap'])
res_no_sector = smf.ols(formula_no_sector, data=df_a5).fit(
    cov_type='cluster', cov_kwds={'groups': df_a5['as_of']})
c1 = res_no_sector.params.get('composite')
p1 = res_no_sector.pvalues.get('composite')
se1 = res_no_sector.bse.get('composite')
print(f'Composite (no sector FE):       coef = {c1:+.4f}   SE = {se1:.4f}   p = {p1:.4f}')
print(f'Composite (with sector FE, ref):coef = {c2:+.4f}   SE = {se2:.4f}   p = {p2:.4f}')
delta = c1 - c2
print(f'\nDelta when sector FE removed: {delta:+.4f}')
if abs(delta) > 0.2 or (p1 < 0.05) != (p2 < 0.05):
    print('FLAG: composite signal is sensitive to sector controls.')
else:
    print('OK: composite signal is robust to sector controls.')


In [ ]:
# AUDIT 6 — Equal-weighted vs cap-weighted quartile portfolio returns
# If cap-weighting drives the result (mega-caps drag Q1 up or down), equal-weighting
# gives every firm-quarter equal voice and isolates the framework signal.

import numpy as np
print('='*70)
print('AUDIT 6 — Equal-weighted vs cap-weighted Q1-Q4 spread (US-only)')
print('='*70)

df_a6 = scored_us.dropna(subset=['forward_return_pct','quartile','market_cap']).copy()

# Equal-weighted quartile means
ew = df_a6.groupby('quartile')['forward_return_pct'].agg(['mean','count']).round(2)
ew.columns = ['ew_mean_pct','n_obs']

# Cap-weighted quartile means
def cap_weighted_mean(g):
    w = g['market_cap'] / g['market_cap'].sum()
    return (g['forward_return_pct'] * w).sum()
cw = df_a6.groupby('quartile').apply(cap_weighted_mean).round(2)
cw.name = 'cw_mean_pct'

combined = ew.join(cw)
combined['cw_minus_ew'] = (combined['cw_mean_pct'] - combined['ew_mean_pct']).round(2)
print(combined.to_string())

# Compute spreads if both Q1 and Q4 (or auto-fallback) are present
quartiles_present = sorted(combined.index.tolist())
top, bottom = quartiles_present[0], quartiles_present[-1]  # alphabetic: Q1 < Q4

ew_spread = combined.loc[top,'ew_mean_pct'] - combined.loc[bottom,'ew_mean_pct']
cw_spread = combined.loc[top,'cw_mean_pct'] - combined.loc[bottom,'cw_mean_pct']
print(f'\n{top}-minus-{bottom} spread:')
print(f'  Equal-weighted: {ew_spread:+.2f}%')
print(f'  Cap-weighted:   {cw_spread:+.2f}%')
print(f'  Difference (cap effect): {cw_spread - ew_spread:+.2f}%')
if abs(cw_spread - ew_spread) > 5:
    print('FLAG: cap-weighting changes the Q1-Q4 spread by more than 5pp. '
          'Size effects materially distort the quartile result.')
else:
    print('OK: spread is robust to weighting scheme.')

combined.to_csv(f'{WORK_DIR}/output/audit6_ew_vs_cw_quartiles.csv')


## Audit summary — how to use in the thesis

**Items flagging concerns** should be discussed in the methodology limitations section. Items returning 'OK' strengthen the thesis claim. Specifically:

- **Audit 1 (top 20 mega-cap concentration):** if flagged, frame Q1 returns as 'partially reflecting concentrated mega-cap exposure rather than pure framework alpha.'
- **Audit 2 (correlations):** report the composite × log_mcap and composite × beta correlations in the methodology section regardless. Examiners will ask.
- **Audits 3-5 (control robustness):** if any flag, the headline regression alone is insufficient. Report results both with and without the problematic control. Best practice: present a 4-column results table showing (1) full controls, (2) no beta, (3) no log_mcap, (4) no sector FE.
- **Audit 6 (weighting scheme):** if cap-weighting matters, report both equal- and cap-weighted spreads. Equal-weighted is the cleaner test of the framework's signal independent of market structure.

All six outputs are saved or reproducible from this notebook. The audit pattern can be re-run on the global panel (`scored`) by changing the data source in each cell.